# 🤖 Notebook 06: Multi-Agent Synthesis (LangChain + Qwen)

**Objective:** Demonstrate the 3-agent LangChain pipeline generating a clinical report.

**Agent Architecture:**
```
PREDICTION
    │
    ├── XAI Results (SHAP + Grad-CAM)
    │       └─── [Agent 1: ExplanationAgent] → Clinical Reasoning Text
    │
    ├── FAISS Results (5 retrieved cases)
    │       └─── [Agent 2: ValidationAgent]  → Evidence-Based Validation
    │
    └── All Outputs ──────────────────────────
            └─── [Agent 3: SummaryAgent]      → Final Clinical Report (Markdown)
```

**LLM:** Qwen/Qwen2.5-72B-Instruct via HuggingFace Inference API

**Sections:**
1. API key verification
2. Initialize all 3 agents
3. Demo Agent 1: ExplanationAgent
4. Demo Agent 2: ValidationAgent
5. Demo Agent 3: SummaryAgent
6. Agent pipeline timing analysis

In [1]:
import sys, os, time
sys.path.insert(0, '..')

import config
from src.agents.explanation_agent import ExplanationAgent
from src.agents.validation_agent  import ValidationAgent
from src.agents.summary_agent     import SummaryAgent

# Set API token in environment
os.environ['HUGGINGFACEHUB_API_TOKEN'] = config.HUGGINGFACEHUB_API_TOKEN

print('=== AGENT CONFIGURATION ===')
print(f'LLM Model:  {config.LLM_MODEL}')
print(f'LLM Mode:   {config.LLM_MODE}')

# Verify HF token is set
if config.HUGGINGFACEHUB_API_TOKEN == 'YOUR_HF_TOKEN_HERE':
    print('⚠️  WARNING: HF_TOKEN not set in config.py!')
    print('   Agents will use fallback mode (rule-based responses)')
else:
    print(f'✅ HF Token: Set ({config.HUGGINGFACEHUB_API_TOKEN[:8]}...)')

=== AGENT CONFIGURATION ===
LLM Model:  Qwen/Qwen2.5-72B-Instruct
LLM Mode:   api
✅ HF Token: Set (hf_pSyJA...)


## 1️⃣ Initialize Agents

In [2]:
print('Initializing Agent 1: ExplanationAgent...')
exp_agent = ExplanationAgent(model_name=config.LLM_MODEL, hf_mode=config.LLM_MODE)
print('✅ ExplanationAgent ready')

print('Initializing Agent 2: ValidationAgent...')
val_agent = ValidationAgent(model_name=config.LLM_MODEL, hf_mode=config.LLM_MODE)
print('✅ ValidationAgent ready')

print('Initializing Agent 3: SummaryAgent...')
sum_agent = SummaryAgent(model_name=config.LLM_MODEL, hf_mode=config.LLM_MODE)
print('✅ SummaryAgent ready')

print('\n=== All 3 agents initialized successfully ===')

Initializing Agent 1: ExplanationAgent...
✅ ExplanationAgent ready
Initializing Agent 2: ValidationAgent...
✅ ValidationAgent ready
Initializing Agent 3: SummaryAgent...
✅ SummaryAgent ready

=== All 3 agents initialized successfully ===


## 2️⃣ Define Sample Patient Case (from MedPix Test Set)

In [3]:
import pandas as pd

df_test = pd.read_csv('../data/processed/test.csv')
sample  = df_test.iloc[0]  # First test sample

# --- Inputs to agents ---
PATIENT_SYNOPSIS  = str(sample['text'])[:400]         # Full clinical text
PREDICTED_LABEL   = sample['label_name']              # Ground truth used as mock prediction
CONFIDENCE        = 0.78                               # Simulated confidence
IMAGE_MODALITY    = str(sample['modality'])
BODY_REGION       = str(sample['location_category'])

# Simulated SHAP results (would normally come from Notebook 04)
SHAP_TOKENS = [
    {'token': 'mass',         'shap_score':  0.0214},
    {'token': 'lesion',       'shap_score':  0.0187},
    {'token': 'enlargement',  'shap_score':  0.0143},
    {'token': 'irregular',    'shap_score':  0.0121},
    {'token': 'calcification','shap_score': -0.0098},
    {'token': 'benign',       'shap_score': -0.0076},
]

# Simulated Grad-CAM region
GRADCAM_REGION = 'upper-right quadrant of the scan, suggesting focal apical lesion'

# Simulated FAISS retrieved cases
RETRIEVED_CASES = [
    {'rank': 1, 'similarity': 0.921, 'image_id': 'MPX1031_synpic12345', 'label_name': PREDICTED_LABEL, 'text': 'Patient with irregular enhancing mass in the right upper lobe.'},
    {'rank': 2, 'similarity': 0.887, 'image_id': 'MPX1045_synpic67890', 'label_name': PREDICTED_LABEL, 'text': 'History of mass with biopsy-confirmed carcinoma.'},
    {'rank': 3, 'similarity': 0.834, 'image_id': 'MPX2001_synpic11111', 'label_name': 'Vascular',       'text': 'Vascular lesion with similar imaging characteristics.'},
    {'rank': 4, 'similarity': 0.812, 'image_id': 'MPX1099_synpic22222', 'label_name': PREDICTED_LABEL, 'text': 'Pulmonary mass in elderly patient, consistent with carcinoma.'},
    {'rank': 5, 'similarity': 0.798, 'image_id': 'MPX1201_synpic33333', 'label_name': PREDICTED_LABEL, 'text': 'Irregular hyperdense lesion in right upper lobe.'},
]

print(f'Case:        {sample["image_id"]}')
print(f'Modality:    {IMAGE_MODALITY}')
print(f'Body region: {BODY_REGION}')
print(f'Pred label:  {PREDICTED_LABEL}')
print(f'Confidence:  {CONFIDENCE:.0%}')
print(f'\nPatient synopsis (first 200 chars):')
print(PATIENT_SYNOPSIS[:200])

Case:        MPX2580_synpic39490
Modality:    MR
Body region: Head
Pred label:  Neoplasm
Confidence:  78%

Patient synopsis (first 200 chars):
52 yo female with complaint of hearing loss. Noncontrast CT:  The patient is status post right suboccipital craniotomy.  There is a subtle area of hypoattenuation off-midline to the right, anterior an


## 3️⃣ Agent 1: ExplanationAgent

In [4]:
print('=' * 60)
print('AGENT 1: ExplanationAgent')
print('=' * 60)

t0 = time.time()

clinical_reasoning = exp_agent.generate_reasoning(
    diagnosis=PREDICTED_LABEL,
    confidence=CONFIDENCE,
    shap_tokens=SHAP_TOKENS,
    gradcam_region=GRADCAM_REGION,
    image_modality=IMAGE_MODALITY,
    body_region=BODY_REGION,
)

t1 = time.time()
print(f'\n[Agent 1 Output — {t1-t0:.1f}s]')
print('-' * 60)
print(clinical_reasoning)

AGENT 1: ExplanationAgent

[Agent 1 Output — 6.0s]
------------------------------------------------------------
The multimodal AI's prediction of neoplasm is supported by the presence of key clinical terms such as 'mass' and 'lesion' in the patient's notes, which are indicative of abnormal tissue growth. The term 'enlargement' further suggests a progressive increase in size, consistent with neoplastic processes. Radiologically, the Grad-CAM heatmap highlighting the upper-right quadrant of the MR scan indicates a focal apical lesion, which aligns with the clinical descriptors of 'irregular' and potentially malignant characteristics. While 'calcification' and 'benign' have negative contributions, the overall pattern of findings strongly points towards a neoplastic process.


## 4️⃣ Agent 2: ValidationAgent

In [5]:
print('=' * 60)
print('AGENT 2: ValidationAgent')
print('=' * 60)

t0 = time.time()

validation_report = val_agent.validate_prediction(
    patient_synopsis=PATIENT_SYNOPSIS,
    predicted_diagnosis=PREDICTED_LABEL,
    confidence=CONFIDENCE,
    retrieved_cases=RETRIEVED_CASES,
)

t1 = time.time()
print(f'\n[Agent 2 Output — {t1-t0:.1f}s]')
print('-' * 60)
print(validation_report)

AGENT 2: ValidationAgent

[Agent 2 Output — 16.1s]
------------------------------------------------------------
**VALIDATION REPORT**

1. **CONSISTENCY CHECK**: Partially
   - The AI prediction of "Neoplasm" is consistent with the majority of the top-5 historical cases (Cases #1, #2, #4, and #5). However, one case (Case #3) suggests a vascular lesion, which introduces some variability.

2. **SUPPORTING EVIDENCE**:
   - **Case #1 (Similarity: 0.921)**: This case involves an irregular enhancing mass, which aligns with the AI's prediction of a neoplasm.
   - **Case #2 (Similarity: 0.887)**: This case also involves a mass with biopsy-confirmed carcinoma, further supporting the neoplasm diagnosis.
   - **Case #4 (Similarity: 0.812)**: Another case of a pulmonary mass consistent with carcinoma, reinforcing the neoplasm prediction.
   - **Case #5 (Similarity: 0.798)**: An irregular hyperdense lesion in the right upper lobe, which is also consistent with a neoplasm.

3. **DISCREPANCIES**:
   -

## 5️⃣ Agent 3: SummaryAgent

In [6]:
print('=' * 60)
print('AGENT 3: SummaryAgent (Final Clinical Report)')
print('=' * 60)

t0 = time.time()

final_report = sum_agent.generate_report(
    patient_synopsis=PATIENT_SYNOPSIS,
    diagnosis=PREDICTED_LABEL,
    confidence=CONFIDENCE,
    clinical_reasoning=clinical_reasoning,
    validation_report=validation_report,
    image_modality=IMAGE_MODALITY,
    body_region=BODY_REGION,
    dominant_modality='image',  # From Integrated Gradients
)

t1 = time.time()
print(f'\n[Agent 3 Output — {t1-t0:.1f}s]')
print('-' * 60)
print(final_report)

AGENT 3: SummaryAgent (Final Clinical Report)

[Agent 3 Output — 14.1s]
------------------------------------------------------------
## 🏥 Patient Summary
A 52-year-old female presents with complaints of hearing loss. Imaging reveals a history of right suboccipital craniotomy and a subtle area of hypoattenuation off-midline to the right, anterior and lateral to the pons and medulla.

## 🔬 Predicted Diagnosis & Confidence
**Diagnosis:** Neoplasm  
**Model Confidence:** 78.0%  
**Primary Evidence Source:** image modality  
The model's confidence indicates a strong but not definitive likelihood of the diagnosis.

## 🧠 Clinical Reasoning (XAI)
The multimodal AI's prediction of neoplasm is supported by the presence of key clinical terms such as 'mass' and 'lesion' in the patient's notes, which suggest abnormal tissue growth. Radiologically, the Grad-CAM heatmap highlights the upper-right quadrant of the MR scan, indicating a region of interest that aligns with the hypoattenuation seen on CT.

In [7]:
# Render the final report as formatted Markdown in the notebook
from IPython.display import display, Markdown
display(Markdown(final_report))

# Save the report to disk
with open('../outputs/sample_clinical_report.md', 'w', encoding='utf-8') as f:
    f.write(final_report)
print('\nClinical report saved to: outputs/sample_clinical_report.md')
print('\n✅ Multi-Agent Synthesis complete. Proceed to Notebook 07: Full Pipeline Demo.')

## 🏥 Patient Summary
A 52-year-old female presents with complaints of hearing loss. Imaging reveals a history of right suboccipital craniotomy and a subtle area of hypoattenuation off-midline to the right, anterior and lateral to the pons and medulla.

## 🔬 Predicted Diagnosis & Confidence
**Diagnosis:** Neoplasm  
**Model Confidence:** 78.0%  
**Primary Evidence Source:** image modality  
The model's confidence indicates a strong but not definitive likelihood of the diagnosis.

## 🧠 Clinical Reasoning (XAI)
The multimodal AI's prediction of neoplasm is supported by the presence of key clinical terms such as 'mass' and 'lesion' in the patient's notes, which suggest abnormal tissue growth. Radiologically, the Grad-CAM heatmap highlights the upper-right quadrant of the MR scan, indicating a region of interest that aligns with the hypoattenuation seen on CT.

## 📁 Historical Case Validation
The AI prediction of "Neoplasm" is partially consistent with historical cases, with the majority (Cases #1, #2, #4, and #5) supporting this diagnosis. One case (Case #3) suggests a vascular lesion, introducing some variability.

## ⚠️ Caveats & Limitations
This AI report provides a probabilistic assessment based on available data and should be interpreted in conjunction with clinical judgment. It does not replace comprehensive clinical evaluation and may have limitations due to data quality and model biases.

## 📋 Recommended Next Steps
1. Review the imaging findings with a neuroradiologist.
2. Consider a biopsy or further imaging (e.g., PET-CT) to confirm the nature of the lesion.
3. Discuss potential treatment options, including surgical intervention, with a multidisciplinary team.

---
*Report generated by Multimodal Clinical AI Pipeline (Trial_2) — FOR PHYSICIAN REVIEW ONLY*

CLINICAL REPORT:


Clinical report saved to: outputs/sample_clinical_report.md

✅ Multi-Agent Synthesis complete. Proceed to Notebook 07: Full Pipeline Demo.
